# Machine Learning Assignment - 2
## Classification Models and Streamlit Deployment

**Dataset:** Breast Cancer Wisconsin (Diagnostic), a public UCI dataset  
**Objective:** Train five classification models and evaluate them using Accuracy, AUC, Precision, Recall, F1 Score and Matthews Correlation Coefficient (MCC).

This notebook also creates the saved models and files needed for the Streamlit application and GitHub repository.

## Step 1 - Install libraries

Run this if the required packages are not already installed. This uses mamba, which you said worked in your BITS Lab environment. If packages are already installed, skip this cell.

In [ ]:
!python -m pip install --upgrade --force-reinstall numpy==1.26.4 scipy==1.11.4 scikit-learn==1.4.2 pandas==2.2.2 matplotlib==3.8.4 seaborn==0.13.2 joblib tabulate

## Step 2 - Import libraries

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)

print("All libraries imported successfully.")

## Step 3 - Create project folders

This creates the main project folder and the folder for the five trained models.

In [ ]:
PROJECT_DIR = "ML_Assignment_2"
MODEL_DIR = os.path.join(PROJECT_DIR, "model")
os.makedirs(MODEL_DIR, exist_ok=True)
print("Project folder:", os.path.abspath(PROJECT_DIR))

## Step 4 - Load the dataset

The scikit-learn copy of the UCI Breast Cancer Wisconsin (Diagnostic) dataset contains 569 records and 30 numeric input features.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
df = dataset.frame.copy()
TARGET = "target"

print("Dataset loaded successfully.")
display(df.head())

## Step 5 - Explore the dataset

In [ ]:
print("Dataset shape:", df.shape)
print("Number of instances:", df.shape[0])
print("Number of input features:", df.drop(columns=[TARGET]).shape[1])
print("\nTarget class counts (0 = malignant, 1 = benign):")
print(df[TARGET].value_counts())
print("\nDataset information:")
df.info()
print("\nSummary statistics:")
display(df.describe())

## Step 6 - Verify dataset requirements

The assignment requires at least 500 instances and at least 12 input features.

In [ ]:
number_of_instances = df.shape[0]
number_of_features = df.drop(columns=[TARGET]).shape[1]

assert number_of_instances >= 500
assert number_of_features >= 12

print("Dataset requirement check passed.")
print("Instances:", number_of_instances)
print("Input features:", number_of_features)

## Step 7 - Check missing values and duplicates

In [ ]:
print("Missing values by column:")
display(df.isnull().sum())
print("Total missing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

## Step 8 - Visualize class distribution

In [ ]:
class_counts = df[TARGET].map({0: "Malignant", 1: "Benign"}).value_counts()
plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title("Target Class Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

## Step 9 - Separate features and target

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
FEATURE_COLUMNS = X.columns.tolist()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Class names:", list(dataset.target_names))

## Step 10 - Split training and testing data

Use 80% for training and 20% for testing. Stratification keeps the class proportions similar.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

## Step 11 - Define the five classification models

Scaling is used for Logistic Regression, KNN and SVM. Imputation is included in each pipeline and is fitted using training data only.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DecisionTreeClassifier(random_state=42))
    ]),
    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Naive Bayes": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", GaussianNB())
    ]),
    "Support Vector Machine": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", SVC(probability=True, random_state=42))
    ])
}
print("Five models are ready.")

## Step 12 - Train all five models

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained successfully.")

## Step 13 - Calculate the six evaluation metrics

Precision, Recall and F1 use weighted averaging. AUC is calculated from the probability of class 1 (benign).

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

## Step 14 - Compare models and identify the top model

The top model below is selected by weighted F1 score. Review the other metrics and confusion matrices as well before making your final conclusion.

In [ ]:
comparison_df = results_df.sort_values("F1", ascending=False).reset_index(drop=True)
display(comparison_df.round(4))
winner_name = comparison_df.loc[0, "ML Model Name"]
print("Top model by weighted F1 score:", winner_name)

## Step 15 - Plot model comparison

In [ ]:
for metric in ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]:
    plt.figure(figsize=(9, 4))
    sns.barplot(data=results_df, x="ML Model Name", y=metric)
    plt.title(f"Model Comparison - {metric}")
    plt.xlabel("Classification Model")
    plt.ylabel(metric)
    plt.xticks(rotation=25, ha="right")
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.show()

## Step 16 - Confusion matrices and classification reports

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print(classification_report(
        y_test, y_pred, target_names=dataset.target_names, zero_division=0
    ))

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=dataset.target_names,
        yticklabels=dataset.target_names
    )
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted label")
    plt.ylabel("Actual label")
    plt.tight_layout()
    plt.show()

## Step 17 - Save the five trained models

Each saved pipeline includes its preprocessing steps, so the Streamlit app can use the same transformations without retraining.

In [ ]:
model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib"
}

for name, filename in model_files.items():
    joblib.dump(models[name], os.path.join(MODEL_DIR, filename))
    print("Saved:", filename)

metadata = {
    "target_column": TARGET,
    "feature_columns": FEATURE_COLUMNS,
    "class_names": list(dataset.target_names),
    "model_files": model_files
}
joblib.dump(metadata, os.path.join(MODEL_DIR, "metadata.joblib"))
print("Models and metadata saved.")

## Step 18 - Create test_data.csv

The uploaded test CSV must contain all input features and the target column. It contains the held-out test rows and is not used to train the models.

In [ ]:
test_data = X_test.copy()
test_data[TARGET] = y_test.to_numpy()

test_data_path = os.path.join(PROJECT_DIR, "test_data.csv")
test_data.to_csv(test_data_path, index=False)

print("Created:", test_data_path)
print("Test data shape:", test_data.shape)
display(test_data.head())

## Step 19 - Create requirements.txt

In [ ]:
requirements_text = 'streamlit\nscikit-learn\nnumpy\npandas\nmatplotlib\nseaborn\njoblib\n'
with open(os.path.join(PROJECT_DIR, "requirements.txt"), "w", encoding="utf-8") as file:
    file.write(requirements_text)

print("requirements.txt created.")

## Step 20 - Create app.py for Streamlit

The app uploads a labelled CSV, lets the user choose a saved classifier, and displays metrics, a confusion matrix and a classification report. It loads saved models and does not retrain them.

In [ ]:

app_code = r"""
import os
import io
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# --------------------------------------------------
# Page configuration
# --------------------------------------------------

st.set_page_config(
    page_title="Machine Learning Model Dashboard",
    page_icon="🧬",
    layout="wide"
)

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(BASE_DIR, "model")

# --------------------------------------------------
# Load metadata
# --------------------------------------------------

metadata_path = os.path.join(MODEL_DIR, "metadata.joblib")

if not os.path.exists(metadata_path):
    st.error("Metadata file not found. Please check the model folder.")
    st.stop()

metadata = joblib.load(metadata_path)

model_files = metadata["model_files"]
model_names = list(model_files.keys())

features = metadata.get("features", metadata.get("feature_columns", []))
target_column = metadata.get(
    "target_column",
    metadata.get("target", "target")
)

# --------------------------------------------------
# Theme selector
# --------------------------------------------------

if "app_theme" not in st.session_state:
    st.session_state.app_theme = "Dark"

top_left, top_right = st.columns([5, 1])

with top_right:
    selected_theme = st.selectbox(
        "Appearance",
        ["Dark", "Light"],
        index=0 if st.session_state.app_theme == "Dark" else 1
    )

st.session_state.app_theme = selected_theme

is_dark = selected_theme == "Dark"

if is_dark:
    bg_colors = ["#07111f", "#101b35", "#17143b", "#082b3b"]
    text_color = "#eaf4ff"
    card_color = "rgba(255,255,255,0.09)"
    border_color = "rgba(255,255,255,0.17)"
    plot_bg = "#101b35"
    plot_fg = "#eaf4ff"
else:
    bg_colors = ["#eff6ff", "#ffffff", "#e0e7ff", "#f0fdfa"]
    text_color = "#172554"
    card_color = "rgba(255,255,255,0.85)"
    border_color = "rgba(30,64,175,0.18)"
    plot_bg = "#ffffff"
    plot_fg = "#172554"

st.markdown(
    f'''
    <style>
    .stApp {{
        background: linear-gradient(
            -45deg,
            {", ".join(bg_colors)}
        );
        background-size: 400% 400%;
        animation: gradientMove 18s ease infinite;
        color: {text_color};
    }}

    @keyframes gradientMove {{
        0% {{background-position: 0% 50%;}}
        50% {{background-position: 100% 50%;}}
        100% {{background-position: 0% 50%;}}
    }}

    [data-testid="stHeader"] {{
        background: transparent;
    }}

    [data-testid="stMetric"] {{
        background: {card_color};
        border: 1px solid {border_color};
        border-radius: 14px;
        padding: 15px;
    }}

    [data-testid="stTabs"] button {{
        font-weight: 600;
    }}

    h1, h2, h3, h4, p, label {{
        color: {text_color};
    }}

    div[data-testid="stDataFrame"] {{
        border: 1px solid {border_color};
        border-radius: 10px;
    }}

    </style>
    ''',
    unsafe_allow_html=True
)

# --------------------------------------------------
# Dashboard heading
# --------------------------------------------------

st.title("🧬 Machine Learning Classification Dashboard")

st.markdown(
    "Interactive analysis, model comparison and performance evaluation "
    "using five machine learning classification algorithms."
)

st.caption("Dataset: Breast Cancer Wisconsin (Diagnostic)")

st.divider()

# --------------------------------------------------
# Load saved models
# --------------------------------------------------

@st.cache_resource
def load_model(model_path):
    return joblib.load(model_path)


loaded_models = {}

for model_name, model_file in model_files.items():
    model_path = os.path.join(MODEL_DIR, model_file)

    if os.path.exists(model_path):
        loaded_models[model_name] = load_model(model_path)

if not loaded_models:
    st.error("No trained model files were found.")
    st.stop()

# --------------------------------------------------
# Dataset selection
# --------------------------------------------------

st.subheader("📂 Dataset")

default_file = os.path.join(BASE_DIR, "test_data.csv")

uploaded_file = st.file_uploader(
    "Upload another labelled test CSV file (optional)",
    type=["csv"]
)

try:
    if uploaded_file is not None:
        data = pd.read_csv(uploaded_file)
        dataset_source = "Uploaded dataset"
    elif os.path.exists(default_file):
        data = pd.read_csv(default_file)
        dataset_source = "Default test_data.csv"
    else:
        st.warning(
            "Default test_data.csv was not found. "
            "Please upload a labelled CSV file."
        )
        st.stop()

except Exception as error:
    st.error(f"Unable to read the CSV file: {error}")
    st.stop()

if not features:
    features = [
        column for column in data.columns
        if column != target_column
    ]

missing_features = [col for col in features if col not in data.columns]

if missing_features:
    st.error(f"Missing feature columns: {missing_features}")
    st.stop()

if target_column not in data.columns:
    st.error(
        f"Target column '{target_column}' is missing. "
        "This dashboard requires labelled data for model evaluation."
    )
    st.stop()

if len(data) == 0:
    st.error("The selected dataset is empty.")
    st.stop()

data = data.copy()
data[features] = data[features].apply(
    pd.to_numeric,
    errors="coerce"
)

data = data.dropna(subset=[target_column])

if len(data) == 0:
    st.error("No valid labelled records are available.")
    st.stop()

X = data[features]
y = data[target_column]

st.caption(f"Dataset source: {dataset_source}")

# --------------------------------------------------
# Evaluate all models
# --------------------------------------------------

def evaluate_model(model, X, y):
    predictions = model.predict(X)

    accuracy = accuracy_score(y, predictions)
    precision = precision_score(
        y, predictions, average="weighted", zero_division=0
    )
    recall = recall_score(
        y, predictions, average="weighted", zero_division=0
    )
    f1 = f1_score(
        y, predictions, average="weighted", zero_division=0
    )
    mcc = matthews_corrcoef(y, predictions)

    auc = np.nan
    probabilities = None

    try:
        if hasattr(model, "predict_proba"):
            probabilities = model.predict_proba(X)

            if len(np.unique(y)) == 2 and probabilities.shape[1] == 2:
                auc = roc_auc_score(y, probabilities[:, 1])

            elif len(np.unique(y)) > 2:
                auc = roc_auc_score(
                    y,
                    probabilities,
                    multi_class="ovr",
                    average="weighted"
                )

        elif hasattr(model, "decision_function"):
            scores = model.decision_function(X)

            if len(np.unique(y)) == 2:
                auc = roc_auc_score(y, scores)

    except Exception:
        auc = np.nan

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "AUC": auc,
        "MCC": mcc,
        "Predictions": predictions,
        "Probabilities": probabilities
    }


@st.cache_data
def evaluate_all_models(X, y, model_names, model_files, model_dir):
    results = {}

    for name in model_names:
        model_path = os.path.join(model_dir, model_files[name])

        if not os.path.exists(model_path):
            continue

        model = joblib.load(model_path)
        results[name] = evaluate_model(model, X, y)

    return results


with st.spinner("Evaluating all five machine learning models..."):
    results = evaluate_all_models(
        X,
        y,
        list(loaded_models.keys()),
        model_files,
        MODEL_DIR
    )

if not results:
    st.error("Unable to evaluate the saved models.")
    st.stop()

# --------------------------------------------------
# Create comparison dataframe
# --------------------------------------------------

metric_names = [
    "Accuracy",
    "AUC",
    "Precision",
    "Recall",
    "F1 Score",
    "MCC"
]

comparison_data = []

for name, result in results.items():
    row = {"Model": name}

    for metric in metric_names:
        row[metric] = result[metric]

    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

# --------------------------------------------------
# Dashboard tabs
# --------------------------------------------------

tab1, tab2, tab3, tab4 = st.tabs([
    "📊 Dataset Overview",
    "🏆 Model Comparison",
    "🔍 Detailed Evaluation",
    "📥 Download Reports"
])

# ==================================================
# TAB 1 - DATASET OVERVIEW
# ==================================================

with tab1:

    st.header("Dataset Overview")

    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Total Records", f"{len(data):,}")
    col2.metric("Total Features", len(features))
    col3.metric("Target Classes", y.nunique())
    col4.metric("Missing Values", int(data[features].isna().sum().sum()))

    st.subheader("Dataset Preview")
    st.dataframe(data.head(10), use_container_width=True)

    left, right = st.columns(2)

    with left:
        st.subheader("Target Class Distribution")

        class_counts = y.value_counts()

        fig, ax = plt.subplots(figsize=(7, 4))
        fig.patch.set_facecolor(plot_bg)
        ax.set_facecolor(plot_bg)

        ax.bar(
            class_counts.index.astype(str),
            class_counts.values
        )

        ax.set_xlabel("Class", color=plot_fg)
        ax.set_ylabel("Number of Records", color=plot_fg)
        ax.set_title("Class Distribution", color=plot_fg)
        ax.tick_params(colors=plot_fg)

        st.pyplot(fig)
        plt.close(fig)

    with right:
        st.subheader("Missing Values by Feature")

        missing = data[features].isna().sum()
        missing = missing[missing > 0].sort_values(ascending=False)

        if len(missing) > 0:
            fig, ax = plt.subplots(figsize=(7, 4))
            fig.patch.set_facecolor(plot_bg)
            ax.set_facecolor(plot_bg)

            ax.barh(missing.index, missing.values)
            ax.set_xlabel("Missing Values", color=plot_fg)
            ax.tick_params(colors=plot_fg)
            ax.set_title("Missing Values", color=plot_fg)

            st.pyplot(fig)
            plt.close(fig)
        else:
            st.success("No missing feature values found.")

    st.subheader("Statistical Summary")
    st.dataframe(
        data[features].describe().T,
        use_container_width=True
    )

    st.subheader("Dataset Information")

    st.write(f"Dataset contains **{len(data)}** valid records.")
    st.write(f"There are **{len(features)}** input features.")
    st.write(f"The target column is **{target_column}**.")
    st.write(f"The dataset contains **{y.nunique()}** target classes.")

# ==================================================
# TAB 2 - MODEL COMPARISON
# ==================================================

with tab2:

    st.header("Comparison of All Five Models")

    st.markdown(
        "The following results are calculated using the same selected "
        "labelled dataset for all available saved models."
    )

    st.subheader("Performance Comparison")

    st.dataframe(
        comparison_df.style.format({
            metric: "{:.4f}" for metric in metric_names
        }),
        use_container_width=True
    )

    st.subheader("Best Model by Metric")

    best_cols = st.columns(3)

    for index, metric in enumerate(metric_names):
        best_index = comparison_df[metric].idxmax()

        if pd.isna(comparison_df.loc[best_index, metric]):
            continue

        best_model = comparison_df.loc[best_index, "Model"]
        best_value = comparison_df.loc[best_index, metric]

        with best_cols[index % 3]:
            st.metric(
                f"Best {metric}",
                best_model,
                f"{best_value:.4f}"
            )

    st.subheader("Select Metrics to Compare")

    selected_metrics = st.multiselect(
        "Choose performance metrics",
        metric_names,
        default=["Accuracy", "Precision", "Recall", "F1 Score"]
    )

    if selected_metrics:
        chart_df = comparison_df.set_index("Model")[selected_metrics]

        st.bar_chart(chart_df, use_container_width=True)

    st.subheader("F1 Score Ranking")

    ranking = comparison_df.sort_values(
        "F1 Score",
        ascending=False
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    fig.patch.set_facecolor(plot_bg)
    ax.set_facecolor(plot_bg)

    ax.bar(
        ranking["Model"],
        ranking["F1 Score"]
    )

    ax.set_ylabel("F1 Score", color=plot_fg)
    ax.set_title("Model Ranking Based on F1 Score", color=plot_fg)
    ax.tick_params(axis="x", colors=plot_fg, rotation=20)
    ax.tick_params(axis="y", colors=plot_fg)

    st.pyplot(fig)
    plt.close(fig)

# ==================================================
# TAB 3 - DETAILED MODEL EVALUATION
# ==================================================

with tab3:

    st.header("Detailed Model Evaluation")

    selected_model = st.selectbox(
        "Select a model for detailed evaluation",
        list(results.keys())
    )

    model_result = results[selected_model]
    predictions = model_result["Predictions"]

    st.subheader(f"{selected_model} Performance")

    metric_cols = st.columns(3)

    display_metrics = [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "AUC",
        "MCC"
    ]

    for index, metric in enumerate(display_metrics):
        value = model_result[metric]

        with metric_cols[index % 3]:
            st.metric(
                metric,
                "N/A" if pd.isna(value) else f"{value:.4f}"
            )

    # Confusion matrix

    st.subheader("Confusion Matrix")

    labels = np.unique(np.concatenate([
        np.asarray(y),
        np.asarray(predictions)
    ]))

    cm = confusion_matrix(y, predictions, labels=labels)

    fig, ax = plt.subplots(figsize=(6, 5))
    fig.patch.set_facecolor(plot_bg)
    ax.set_facecolor(plot_bg)

    image = ax.imshow(cm, interpolation="nearest")

    ax.set_title("Confusion Matrix", color=plot_fg)
    ax.set_xlabel("Predicted Label", color=plot_fg)
    ax.set_ylabel("Actual Label", color=plot_fg)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))

    ax.set_xticklabels(labels, color=plot_fg)
    ax.set_yticklabels(labels, color=plot_fg)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black"
            )

    fig.colorbar(image, ax=ax)
    st.pyplot(fig)
    plt.close(fig)

    # Classification report

    st.subheader("Classification Report")

    report = classification_report(
        y,
        predictions,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).T
    st.dataframe(report_df, use_container_width=True)

    # ROC curve

    st.subheader("ROC Curve")

    probabilities = model_result["Probabilities"]

    if len(np.unique(y)) == 2 and probabilities is not None:
        try:
            model = loaded_models[selected_model]
            classes = model.classes_

            positive_class = classes[1]
            positive_index = list(classes).index(positive_class)

            y_binary = (np.asarray(y) == positive_class).astype(int)

            fpr, tpr, thresholds = roc_curve(
                y_binary,
                probabilities[:, positive_index]
            )

            fig, ax = plt.subplots(figsize=(7, 5))
            fig.patch.set_facecolor(plot_bg)
            ax.set_facecolor(plot_bg)

            ax.plot(fpr, tpr, label="ROC Curve")
            ax.plot([0, 1], [0, 1], linestyle="--")

            ax.set_xlabel("False Positive Rate", color=plot_fg)
            ax.set_ylabel("True Positive Rate", color=plot_fg)
            ax.set_title("ROC Curve", color=plot_fg)
            ax.tick_params(colors=plot_fg)
            ax.legend()

            st.pyplot(fig)
            plt.close(fig)

        except Exception as error:
            st.info(f"ROC curve could not be displayed: {error}")
    else:
        st.info(
            "ROC curve is displayed for binary classification models "
            "with probability predictions."
        )

    # Prediction preview

    st.subheader("Actual vs Predicted Values")

    prediction_preview = pd.DataFrame({
        "Actual": np.asarray(y),
        "Predicted": np.asarray(predictions)
    })

    prediction_preview["Correct Prediction"] = (
        prediction_preview["Actual"]
        == prediction_preview["Predicted"]
    )

    st.dataframe(
        prediction_preview.head(20),
        use_container_width=True
    )

    correct_count = int(
        prediction_preview["Correct Prediction"].sum()
    )

    incorrect_count = len(prediction_preview) - correct_count

    c1, c2 = st.columns(2)
    c1.metric("Correct Predictions", correct_count)
    c2.metric("Incorrect Predictions", incorrect_count)

# ==================================================
# TAB 4 - DOWNLOAD REPORTS
# ==================================================

with tab4:

    st.header("Download Evaluation Reports")

    st.write(
        "Download the model comparison, prediction results "
        "and detailed evaluation report."
    )

    # Comparison CSV

    comparison_csv = comparison_df.to_csv(index=False).encode("utf-8")

    st.download_button(
        "📊 Download Model Comparison CSV",
        data=comparison_csv,
        file_name="model_comparison_report.csv",
        mime="text/csv",
        use_container_width=True
    )

    # All predictions CSV

    all_predictions = pd.DataFrame({
        "Actual": np.asarray(y)
    })

    for name, result in results.items():
        all_predictions[name + "_Prediction"] = result["Predictions"]

    predictions_csv = all_predictions.to_csv(
        index=False
    ).encode("utf-8")

    st.download_button(
        "📥 Download All Model Predictions CSV",
        data=predictions_csv,
        file_name="all_model_predictions.csv",
        mime="text/csv",
        use_container_width=True
    )

    # Detailed text report

    report_lines = []

    report_lines.append("MACHINE LEARNING MODEL EVALUATION REPORT")
    report_lines.append("=" * 55)
    report_lines.append(f"Dataset source: {dataset_source}")
    report_lines.append(f"Total records: {len(data)}")
    report_lines.append(f"Total features: {len(features)}")
    report_lines.append(f"Target column: {target_column}")
    report_lines.append("")

    report_lines.append("MODEL PERFORMANCE COMPARISON")
    report_lines.append("-" * 55)
    report_lines.append(comparison_df.to_string(index=False))
    report_lines.append("")

    for name, result in results.items():

        report_lines.append("=" * 55)
        report_lines.append(f"MODEL: {name}")
        report_lines.append("=" * 55)

        for metric in metric_names:
            value = result[metric]
            report_lines.append(
                f"{metric}: "
                f"{'N/A' if pd.isna(value) else f'{value:.4f}'}"
            )

        report_lines.append("")
        report_lines.append("CLASSIFICATION REPORT")
        report_lines.append(
            classification_report(
                y,
                result["Predictions"],
                zero_division=0
            )
        )

        report_lines.append("CONFUSION MATRIX")
        report_lines.append(
            str(confusion_matrix(y, result["Predictions"]))
        )

        report_lines.append("")

    full_report = "\n".join(report_lines)

    st.download_button(
        "📄 Download Detailed Evaluation Report (TXT)",
        data=full_report.encode("utf-8"),
        file_name="detailed_model_evaluation_report.txt",
        mime="text/plain",
        use_container_width=True
    )

    # Selected model prediction CSV

    report_model = st.selectbox(
        "Select model for individual prediction report",
        list(results.keys()),
        key="download_model"
    )

    selected_predictions = pd.DataFrame({
        "Actual": np.asarray(y),
        "Predicted": results[report_model]["Predictions"]
    })

    selected_predictions["Correct"] = (
        selected_predictions["Actual"]
        == selected_predictions["Predicted"]
    )

    selected_csv = selected_predictions.to_csv(
        index=False
    ).encode("utf-8")

    st.download_button(
        "📑 Download Selected Model Predictions",
        data=selected_csv,
        file_name=report_model.replace(" ", "_") + "_predictions.csv",
        mime="text/csv",
        use_container_width=True
    )

# --------------------------------------------------
# Footer
# --------------------------------------------------

st.divider()

st.markdown(
    "### Machine Learning Assignment - 2"
)

st.caption(
    "Classification Models | Model Evaluation | "
    "Performance Comparison | Streamlit Deployment"
)
"""

# --------------------------------------------------
# Save Streamlit application
# --------------------------------------------------

app_path = os.path.join(PROJECT_DIR, "app.py")

with open(app_path, "w", encoding="utf-8") as file:
    file.write(app_code)

print("SUCCESS: Updated Streamlit app.py created.")
print("Location:", app_path)
print("Features included:")
print("1. Light and Dark theme selector")
print("2. Animated gradient background")
print("3. Automatic default test CSV loading")
print("4. Dataset overview")
print("5. Comparison of all five models")
print("6. Detailed model evaluation")
print("7. Confusion matrix and ROC curve")
print("8. Downloadable evaluation reports")


## Step 21 - Create README.md

The README includes the required problem statement, dataset description, repository link placeholder, comparison table, model observations and overall winner. Update the GitHub link after creating your repository.

In [ ]:

# Prepare the comparison table
readme_results = results_df.copy()

readme_results = readme_results.rename(columns={
    "F1": "F1 Score"
})

metric_columns = [
    "Accuracy", "AUC", "Precision",
    "Recall", "F1 Score", "MCC"
]

readme_results[metric_columns] = (
    readme_results[metric_columns].round(4)
)

comparison_table = readme_results[
    ["ML Model Name"] + metric_columns
].to_markdown(index=False)


# Prepare short observations based on actual model results
observation_text = {
    "Logistic Regression":
        "Performs well with good overall accuracy and balanced results.",

    "Decision Tree":
        "Provides good classification results but may be less consistent.",

    "KNN":
        "Provides reasonable results based on similarity between data points.",

    "Naive Bayes":
        "Offers simple and fast classification with reasonable performance.",

    "Support Vector Machine":
        "Provides strong classification performance with balanced results."
}

observation_rows = []

for model_name in readme_results["ML Model Name"]:

    observation = observation_text.get(
        model_name,
        "Provides classification results based on the selected dataset."
    )

    observation_rows.append(
        f"| {model_name} | {observation} |"
    )

observations_table = "\n".join(observation_rows)


# Select the overall winner using weighted F1 score
winner_name = (
    readme_results
    .sort_values("F1 Score", ascending=False)
    .iloc[0]["ML Model Name"]
)


# Create README content
readme_text = f"""# Machine Learning Assignment - 2
## Classification Models and Evaluation

## a. Problem Statement

The objective of this assignment is to implement five machine learning classification models on the selected dataset and compare their performance using six evaluation metrics.

## b. Dataset Description

- **Dataset:** Breast Cancer Wisconsin (Diagnostic)
- **Source:** UCI Machine Learning Repository
- **Total Instances:** {df.shape[0]}
- **Number of Input Features:** {len(FEATURE_COLUMNS)}
- **Problem Type:** Binary Classification
- **Target Variable:** Diagnosis (Malignant or Benign)

## c. GitHub Repository Link

**GitHub Repository:** Add your actual GitHub repository URL here.

## d. Models Used

The following five classification models were implemented and evaluated on the same dataset:

1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbors (KNN)
4. Gaussian Naive Bayes
5. Support Vector Machine (SVM)

### Model Performance Comparison

{comparison_table}

### Observations on Model Performance

| ML Model Name | Observation about model performance |
|---|---|
{observations_table}

### Overall Winner for the Dataset

**{winner_name}**

The overall winner is selected based on the highest weighted F1 score among the five models.

## Streamlit Application

The Streamlit application allows users to upload test data, select a classification model and view evaluation metrics, confusion matrix and classification report.

The application loads the saved models and evaluates the uploaded test data without retraining.
"""

# Save README.md
readme_path = os.path.join(PROJECT_DIR, "README.md")

with open(readme_path, "w", encoding="utf-8") as file:
    file.write(readme_text)

print("README.md created successfully.")
print("Location:", readme_path)
print("Overall winner:", winner_name)

## Step 22 - Check the generated project structure

In [ ]:
for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    for filename in sorted(files):
        print(f"{indent}    {filename}")

## Step 23 - Final file and result checks

In [ ]:
required_files = [
    "app.py", "requirements.txt", "README.md", "test_data.csv",
    os.path.join("model", "logistic_regression.joblib"),
    os.path.join("model", "decision_tree.joblib"),
    os.path.join("model", "knn.joblib"),
    os.path.join("model", "naive_bayes.joblib"),
    os.path.join("model", "svm.joblib"),
    os.path.join("model", "metadata.joblib")
]

missing_files = [
    name for name in required_files
    if not os.path.exists(os.path.join(PROJECT_DIR, name))
]

assert not missing_files, f"Missing files: {missing_files}"
assert len(results_df) == 5
assert results_df[["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]].notna().all().all()
assert len(test_data) == len(X_test)

print("SUCCESS: All required project files have been created.")
print("SUCCESS: All five models were evaluated.")
print("SUCCESS: Test data checks passed.")